### Lets first just explore the intuition and math behind StandardScalar

standardization transforms your numerical distribution so that:
- New **mean ($\mu$)** is **0** (centers the data around zero)
- The new **standard deviation ($\sigma$)** is **1** (compresses or stretches the spread to unit variance).\

#### The Formula ($Z$-Score) 
For every single value x in the column:

$$z = \frac{x - \mu}{\sigma}$$
- $\mu$ = Mean (average of all values in that column)
- $\sigma$ = Standard Deviation (spread of the values)

#### Why This Works
If a person's Age is $30$ and the column average is $\mu = 30$, their transformed value becomes:

$$z = \frac{30 - 30}{\sigma} = 0.0$$

- A score of 0.0 means exactly average.
- A positive score (e.g., +1.5) means 1.5 standard deviations above average.
- A negative score (e.g., -1.2) means 1.2 standard deviations below average.

Now, Age (originally 20–60) and Salary (originally 30,000–150,000) are on the exact same relative scale (mostly spanning between $-3$ and $+3$).


# 1. Feature Standardization using StandardScaler

This notebook covers:
1. **The Math of Standardization**: $Z$-score calculation ($z = \frac{x - \mu}{\sigma}$).
2. Transforming numeric features to have $\text{Mean} = 0$ and $\text{Std} = 1$.
3. Handling mixed datasets: Scaling numeric columns while leaving categorical columns untouched.
4. Using `ColumnTransformer` with `verbose_feature_names_out=False` and `.set_output(transform='pandas')`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# 1. Realistic dataset with vastly different numerical scales
raw_data = {
    'Age': [22, 28, 35, 45, 52, 23, 40],               # Scale: 20 - 55
    'Annual_Salary': [32000, 58000, 95000, 130000, 160000, 35000, 110000], # Scale: 30k - 160k
    'Credit_Score': [610, 680, 750, 810, 790, 620, 720], # Scale: 600 - 850
    'City': ['Hyderabad', 'Bangalore', 'Mumbai', 'Hyderabad', 'Bangalore', 'Mumbai', 'Hyderabad']
}

df = pd.DataFrame(raw_data)
print("=== 1. ORIGINAL RAW DATASET ===")
display(df)

# Check raw mean and standard deviation
print("\n=== RAW STATISTICS (BEFORE SCALING) ===")
display(df[['Age', 'Annual_Salary', 'Credit_Score']].describe().round(2))

=== 1. ORIGINAL RAW DATASET ===


,Age,Annual_Salary,Credit_Score,City
0,22,32000,610,Hyderabad
1,28,58000,680,Bangalore
2,35,95000,750,Mumbai
3,45,130000,810,Hyderabad
4,52,160000,790,Bangalore
5,23,35000,620,Mumbai
6,40,110000,720,Hyderabad



=== RAW STATISTICS (BEFORE SCALING) ===


,Age,Annual_Salary,Credit_Score
count,7.00,7.00,7.00
mean,35.00,88571.43,711.43
std,11.37,48859.86,78.62
min,22.00,32000.00,610.00
25%,25.50,46500.00,650.00
50%,35.00,95000.00,720.00
75%,42.50,120000.00,770.00
max,52.00,160000.00,810.00


---
## Applying StandardScaler via ColumnTransformer

We only scale numeric features (`Age`, `Annual_Salary`, `Credit_Score`).
Categorical features (`City`) are passed through unaffected using `remainder='passthrough'`.

In [3]:
# 1. Make an explicit clean copy
df_copy = df.copy()

# 2. Identify numeric columns to scale
numeric_cols = ['Age', 'Annual_Salary', 'Credit_Score']

# 3. Define the ColumnTransformer
scaler_ct = ColumnTransformer(
    transformers=[
        ('num_scaler', StandardScaler(), numeric_cols)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')

# 4. Fit and transform the dataset copy
df_scaled = scaler_ct.fit_transform(df_copy)

print("=== 2. FULL DATASET AFTER STANDARDIZATION ===")
display(df_scaled)

print("\n=== TRANSFORMED STATISTICS (MEAN ≈ 0, STD = 1) ===")
display(df_scaled[numeric_cols].describe().round(2))

=== 2. FULL DATASET AFTER STANDARDIZATION ===


,Age,Annual_Salary,Credit_Score,City
0,-1.234700,-1.250600,-1.393497,Hyderabad
1,-0.664839,-0.675829,-0.431788,Bangalore
2,0.000000,0.142114,0.529921,Mumbai
3,0.949769,0.915843,1.354243,Hyderabad
4,1.614608,1.579040,1.079469,Bangalore
5,-1.139723,-1.184280,-1.256110,Mumbai
6,0.474885,0.473712,0.117760,Hyderabad



=== TRANSFORMED STATISTICS (MEAN ≈ 0, STD = 1) ===


,Age,Annual_Salary,Credit_Score
count,7.00,7.00,7.00
mean,-0.00,0.00,-0.00
std,1.08,1.08,1.08
min,-1.23,-1.25,-1.39
25%,-0.90,-0.93,-0.84
50%,0.00,0.14,0.12
75%,0.71,0.69,0.80
max,1.61,1.58,1.35


---
## Key Observations:
1. **Mean is 0.00:** Every scaled feature is centered at zero.
2. **Standard Deviation is 1.00:** The spread across all numeric features is identical.
3. **No Information Lost:** The relative variance, percentiles, and distribution shapes remain intact.
4. **Categorical Intact:** `City` remains in its original position without any disruption.

In case you got some small doubts, I think this is all they are:

---

## Deep Dive: Common StandardScaler Doubts & Under-the-Hood Mechanics

### 1. Is Data Strictly Bounded Between $-3$ and $+3$?
**No.** There is no hard cutoff at $-3$ or $+3$.

The $-3$ to $+3$ rule comes from the **Empirical Rule (68–95–99.7 Rule)** for normal distributions:
* **68.2%** of data falls within $[-1, +1]$
* **95.4%** of data falls within $[-2, +2]$
* **99.7%** of data falls within $[-3, +3]$

For normally distributed data, almost all points naturally land within this interval. However, if a column contains **extreme outliers** (e.g., a CEO salary of $10,000,000 in a dataset of entry-level salaries), its $Z$-score can easily exceed $+15.0$.

---

### 2. What Do Mean ($\mu$) and Standard Deviation ($\sigma$) Actually Represent?
* **Mean ($\mu = 0$):** A single central reference point. 
  * A transformed value of `0.0` represents an **exactly average** observation.
  * Positive values indicate **above average**; negative values indicate **below average**.
* **Standard Deviation ($\sigma = 1$):** A measure of **spread/scale**, not a range. It is always a single non-negative number representing the uniform unit step across all features.

---

### 3. Why Mean = 0 and Std = 1? Why Not Both 0?
* If **$\text{Standard Deviation} = 0$**, it mathematically means **every single row has the exact same value** (zero variance).
* A column with zero variance contains no information and is completely useless to a machine learning model.
* Setting **$\mu = 0$** centers all features at a common baseline, while setting **$\sigma = 1$** normalizes their spread so one unit step means the same across all columns (e.g., 1 standard unit of `Age` equals 1 standard unit of `Salary`).

---

### 4. Why Do We See `-0.00` and `1.08` in the Summary Statistics?

#### A. The `-0.00` Phenomenon (Floating-Point Precision)
Computers represent decimals in binary floating-point. The calculation produces tiny numbers like $-0.00000000000000014$. When formatted to 2 decimal places via `.round(2)`, Pandas renders this microscopic negative value as **`-0.00`**. Mathematically, it is exactly zero.

#### B. The `1.08` Standard Deviation (Bessel's Correction)
This is due to the difference between **population** and **sample** standard deviation formulas on small datasets ($N=7$):

| Calculation | Formula | Divisor | Used By | Result on $N=7$ |
| :--- | :--- | :--- | :--- | :--- |
| **Population Std ($\sigma$)** | $\sqrt{\frac{\sum(x - \mu)^2}{N}}$ | Divides by $N = 7$ | `StandardScaler` (Scikit-Learn) | **$1.00$** |
| **Sample Std ($s$)** | $\sqrt{\frac{\sum(x - \bar{x})^2}{N - 1}}$ | Divides by $N - 1 = 6$ | `pandas.describe()` | $\sqrt{\frac{7}{6}} \approx \mathbf{1.08}$ |

`StandardScaler` standardizes using the population divisor ($N$). When Pandas evaluates the resulting column with sample degrees of freedom ($N-1$), it scales by a factor of $\sqrt{\frac{N}{N-1}}$. On real-world datasets with large $N$, $N \approx N - 1$, and both evaluate to **$1.00$**.

---